# Acoustic ID — Module 2B: Noise Simulator
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Reads clean `.wav` files from `acoustic_signals/` (Module 2 output)
- Synthesises 6 real-world urban noise profiles entirely in software
- Generates 24 noisy versions per vehicle (6 profiles × 4 dB levels)
- Saves all noisy files to `noisy_signals/` folder
- Skips files that already exist — safe to re-run multiple times without errors
- Zero database interaction — purely a filesystem operation

**Output consumed by:** Module 3 reads from `noisy_signals/` for reliability testing

**Why synthetic noise:** Fully reproducible, mathematically precise, no external dataset dependencies. Fixed random seed ensures identical output every run.

**Libraries:** `numpy`, `scipy`, `soundfile` — same as Module 2, no new installs required.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import soundfile as sf
import os
import glob
from scipy import signal as sp_signal

## Step 2: Configuration

**Only these two lines need to change between runs.**

All other constants are fixed parameters that Module 3 depends on — do not change them.

In [ ]:
# ── USER CONFIGURATION — change only these two lines ─────────────────────────
SIGNALS_DIR = "acoustic_signals"    # clean .wav input folder (Module 2 output)
NOISY_DIR   = "noisy_signals"       # noisy .wav output folder (Module 3 input)
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_RATE = 96000             # must match Module 2 and Module 3
SEED        = 42                # fixed seed — guarantees identical output every run
DB_LEVELS   = [60, 70, 80, 90] # dB SPL noise levels to simulate

# Noise profile registry — maps profile name to synthesiser function (defined in Step 4)
# Populated after Step 4 is executed
SYNTHESISERS = {}

notebook_dir = os.getcwd()
signals_dir  = os.path.join(notebook_dir, SIGNALS_DIR)
noisy_dir    = os.path.join(notebook_dir, NOISY_DIR)

os.makedirs(noisy_dir, exist_ok=True)

print(f"Clean signals : '{signals_dir}'")
print(f"Noisy output  : '{noisy_dir}'")
print(f"Noise levels  : {DB_LEVELS} dB")
print(f"Profiles      : 6 (registered after Step 4)")
print(f"Files per vehicle: {len(DB_LEVELS) * 6} (6 profiles × {len(DB_LEVELS)} dB levels)")

## Step 3: Core Utilities

### `bandlimited_noise`
Generates Gaussian white noise filtered to a specific frequency band using a 4th-order Butterworth bandpass filter in SOS (second-order sections) form. SOS form is used instead of the standard BA form because BA-form filters at low frequencies (e.g. 60–200 Hz DJ bass) cause numerical overflow — a known issue with `scipy.signal.lfilter`. SOS form is numerically stable at all frequencies.

Output is always normalised to unit peak before returning, so `scale_to_db` operates on a consistent amplitude reference.

### `measure_db` and `scale_to_db`
RMS-based dB measurement and precise scaling. Noise is scaled to the exact target dB level **before** mixing with the clean signal — so the dB value refers to the noise floor alone, not the combined output.

### `mix_noise`
Adds scaled noise to the clean OOK signal and clips the result to `[-1.0, 1.0]`. Clipping prevents WAV file artefacts and ensures Module 3 can load every file without error.

In [ ]:
def bandlimited_noise(
    n_samples : int,
    low_hz    : float,
    high_hz   : float,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Generate Gaussian noise bandlimited to [low_hz, high_hz] Hz.
    Uses SOS-form Butterworth filter for numerical stability.
    Output is normalised to unit peak amplitude.

    Args:
        n_samples (int)   : Number of samples to generate
        low_hz    (float) : Lower cutoff frequency in Hz
        high_hz   (float) : Upper cutoff frequency in Hz
        sr        (int)   : Sample rate in Hz
        rng       (np.random.Generator): Seeded random generator

    Returns:
        np.ndarray: float32 noise array, peak-normalised to [-1.0, 1.0]
    """
    noise = rng.standard_normal(n_samples).astype(np.float64)
    nyq   = sr / 2.0
    low   = np.clip(low_hz  / nyq, 1e-4, 0.9999)
    high  = np.clip(high_hz / nyq, 1e-4, 0.9999)
    if low >= high:
        peak = np.max(np.abs(noise))
        return (noise / peak if peak > 0 else noise).astype(np.float32)
    sos      = sp_signal.butter(4, [low, high], btype="band", output="sos")
    filtered = sp_signal.sosfilt(sos, noise)
    peak     = np.max(np.abs(filtered))
    return (filtered / peak if peak > 0 else filtered).astype(np.float32)


def measure_db(signal: np.ndarray) -> float:
    """
    Compute RMS power of a signal in dB.

    Args:
        signal (np.ndarray): Audio signal

    Returns:
        float: RMS level in dB, or -inf if signal is silent
    """
    rms = np.sqrt(np.mean(signal.astype(np.float64) ** 2))
    return 20 * np.log10(rms) if rms > 1e-12 else -np.inf


def scale_to_db(signal: np.ndarray, target_db: float) -> np.ndarray:
    """
    Scale a signal to a target RMS dB level.

    Args:
        signal    (np.ndarray): Input signal (should be peak-normalised)
        target_db (float)     : Target RMS level in dB

    Returns:
        np.ndarray: float32 signal scaled to target_db
    """
    current = measure_db(signal)
    if current == -np.inf:
        return signal.copy()
    gain = 10 ** ((target_db - current) / 20)
    return (signal.astype(np.float64) * gain).astype(np.float32)


def mix_noise(
    clean     : np.ndarray,
    noise     : np.ndarray,
    target_db : float
) -> np.ndarray:
    """
    Scale noise to target dB, add to clean signal, clip to [-1.0, 1.0].

    Args:
        clean     (np.ndarray): Clean OOK signal from Module 2
        noise     (np.ndarray): Synthesised noise profile (peak-normalised)
        target_db (float)     : Target noise floor level in dB SPL

    Returns:
        np.ndarray: float32 mixed signal, clipped to [-1.0, 1.0]
    """
    noise_scaled = scale_to_db(noise, target_db)
    mixed        = clean.astype(np.float64) + noise_scaled.astype(np.float64)
    return np.clip(mixed, -1.0, 1.0).astype(np.float32)

## Step 4: Noise Profile Synthesisers

Each function generates a synthetic noise signal matching the spectral and temporal characteristics of a real urban noise source. All functions have the same signature:
`synthesise_X(n_samples, sr, rng) -> np.float32 array`

All outputs are peak-normalised to `[-1.0, 1.0]` before returning.

| Profile | Frequency range | Modulation | Real-world scenario |
|---|---|---|---|
| `traffic` | 80–6000 Hz (engine + tyre + road) | Slow swell 0.2 Hz | Dense urban traffic, rush hour |
| `rain` | Full spectrum, rolloff above 10 kHz | None — steady | Monsoon, heavy downpour |
| `dj` | 60–2000 Hz (bass-heavy) | None — sustained | DJ event, loud music outside hospital |
| `procession` | 80–8000 Hz broadband | Rhythmic bursts at 2 Hz | Dhol/band wedding procession |
| `hawker` | 300–3400 Hz voice band | Speech-rhythm AM at 5 Hz | Street vendor with loudspeaker |
| `crowd` | 300–4000 Hz multi-layer | Irregular overlapping AM | Rally, protest, market crowd |

In [ ]:
def synthesise_traffic(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Urban traffic noise: layered engine rumble (80-500 Hz), tyre noise
    (500-2000 Hz), and high-frequency road surface noise (2-6 kHz).
    Slow amplitude swell at 0.2 Hz simulates passing vehicle density.
    Scenario: dense urban traffic, rush hour, major intersection.
    """
    engine = bandlimited_noise(n_samples, 80,   500,  sr, rng) * 0.6
    tyre   = bandlimited_noise(n_samples, 500,  2000, sr, rng) * 0.3
    road   = bandlimited_noise(n_samples, 2000, 6000, sr, rng) * 0.1
    mixed  = (engine + tyre + road).astype(np.float64)
    t      = np.arange(n_samples) / sr
    swell  = 0.7 + 0.3 * np.sin(2 * np.pi * 0.2 * t)
    mixed *= swell
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_rain(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Rain noise: full-spectrum white noise with gentle high-frequency rolloff
    above 10 kHz (models droplet impact physics). Closest to true AWGN —
    the most spectrally uniform and hardest to filter out.
    Scenario: heavy monsoon, extreme downpour with wind.
    """
    white  = rng.standard_normal(n_samples).astype(np.float64)
    sos    = sp_signal.butter(2, 10000 / (sr / 2), btype="low", output="sos")
    result = sp_signal.sosfilt(sos, white)
    peak   = np.max(np.abs(result))
    return (result / peak if peak > 0 else result).astype(np.float32)


def synthesise_dj(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    DJ/loud music noise: heavy bass content (60-200 Hz) dominating,
    mid-range (200-800 Hz) and harmonic content (800-2000 Hz) at lower levels.
    Sustained — no temporal modulation, simulating continuous music playback.
    Scenario: DJ event or loud speaker system adjacent to silent zone.
    """
    bass   = bandlimited_noise(n_samples, 60,  200,  sr, rng) * 0.7
    mid    = bandlimited_noise(n_samples, 200, 800,  sr, rng) * 0.2
    harm   = bandlimited_noise(n_samples, 800, 2000, sr, rng) * 0.1
    mixed  = (bass + mid + harm).astype(np.float64)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_procession(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Procession/dhol noise: broadband noise (80-8000 Hz) with a rhythmic
    amplitude envelope at 2 Hz (dhol beat pattern) plus an off-beat accent.
    Simulates the energy burst pattern of dhol or band instruments.
    Scenario: wedding procession or religious band near hospital gate.
    """
    broad  = bandlimited_noise(n_samples, 80, 8000, sr, rng).astype(np.float64)
    t      = np.arange(n_samples) / sr
    beat   = np.abs(np.sin(2 * np.pi * 2.0 * t)) ** 0.3
    accent = 0.4 * np.abs(np.sin(2 * np.pi * 2.0 * t + np.pi * 0.6)) ** 0.5
    env    = np.clip(beat + accent, 0.1, 1.0)
    mixed  = broad * env
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_hawker(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Street hawker/vendor noise: voice-band noise (300-3400 Hz) with
    amplitude modulation at 5 Hz and 1.3 Hz (simulates speech rhythm
    and sentence cadence). Intermittent — dips to 5% amplitude between calls.
    Scenario: street vendor with loudspeaker near school or court.
    """
    voice  = bandlimited_noise(n_samples, 300, 3400, sr, rng).astype(np.float64)
    t      = np.arange(n_samples) / sr
    mod    = 0.5 + 0.5 * np.sin(2 * np.pi * 5.0 * t) * np.sin(2 * np.pi * 1.3 * t)
    mod    = np.clip(mod, 0.05, 1.0)
    mixed  = voice * mod
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_crowd(
    n_samples : int,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Crowd noise: multiple overlapping voice-band layers (300-4000 Hz)
    with an irregular combined amplitude pattern from two slow sinusoids.
    Simulates the unpredictable energy bursts of a large gathering.
    Scenario: protest, rally, or dense market crowd near silent zone.
    """
    v1     = bandlimited_noise(n_samples, 300,  2000, sr, rng) * 0.4
    v2     = bandlimited_noise(n_samples, 500,  3400, sr, rng) * 0.3
    chat   = bandlimited_noise(n_samples, 1000, 4000, sr, rng) * 0.2
    amb    = bandlimited_noise(n_samples, 200,  800,  sr, rng) * 0.1
    mixed  = (v1 + v2 + chat + amb).astype(np.float64)
    t      = np.arange(n_samples) / sr
    irr    = 0.5 + 0.3 * np.sin(2 * np.pi * 0.7 * t) + 0.2 * np.sin(2 * np.pi * 1.9 * t)
    mixed *= np.clip(irr, 0.1, 1.0)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


# Register all synthesisers — used by batch encoder in Step 6
SYNTHESISERS = {
    "traffic"    : synthesise_traffic,
    "rain"       : synthesise_rain,
    "dj"         : synthesise_dj,
    "procession" : synthesise_procession,
    "hawker"     : synthesise_hawker,
    "crowd"      : synthesise_crowd,
}

print(f"Registered profiles: {list(SYNTHESISERS.keys())}")
print(f"Total files per vehicle: {len(SYNTHESISERS) * len(DB_LEVELS)}")

## Step 5: Skip Logic

`needs_generation()` checks whether a specific noisy file already exists on disk.
This is a pure `os.path.exists()` check — no database, no complex state.

Running the module multiple times is safe:
- If all 24 files exist for a vehicle → all 24 skipped, zero re-computation
- If some files are missing (deleted, partial run, corrupted) → only missing files regenerated
- If the clean source WAV is missing → vehicle is skipped entirely with a clear error message

No error is raised for any of these cases — all failures are collected and reported in the summary.

In [ ]:
def needs_generation(
    plate    : str,
    profile  : str,
    db_level : int,
    noisy_dir: str
) -> bool:
    """
    Check whether a noisy WAV file needs to be generated.

    Args:
        plate     (str): Vehicle plate number
        profile   (str): Noise profile name (e.g. "traffic")
        db_level  (int): Target dB level (e.g. 60)
        noisy_dir (str): Path to noisy_signals output folder

    Returns:
        bool: True if file does not exist and needs to be generated
    """
    fname = f"{plate}_{profile}_{db_level}db.wav"
    return not os.path.exists(os.path.join(noisy_dir, fname))

## Step 6: Single Vehicle Noise Generator

`generate_noisy_files()` processes one vehicle — loads its clean WAV, generates all 24 noisy versions, saves each to `noisy_signals/`.

**Fallback handling:**
- Clean WAV file not found → returns immediately with descriptive error, no crash
- WAV file unreadable (corrupt) → returns with error, does not abort other vehicles
- Sample rate mismatch → caught and reported before any processing begins
- Individual file write failure → recorded in `failed` list, remaining files continue

The `rng` is passed in from the batch runner so the seed is consistent across all vehicles.

In [ ]:
def generate_noisy_files(
    plate         : str,
    clean_wav_path: str,
    noisy_dir     : str,
    rng           : np.random.Generator
) -> dict:
    """
    Generate all 24 noisy WAV files for a single vehicle.

    Args:
        plate          (str): Vehicle plate number
        clean_wav_path (str): Path to clean .wav file from Module 2
        noisy_dir      (str): Output folder for noisy files
        rng (np.random.Generator): Seeded RNG — passed from batch runner

    Returns:
        dict: {
            "plate"   : str,
            "encoded" : list of filenames successfully written,
            "skipped" : list of filenames that already existed,
            "failed"  : list of dicts {file, error} for any failures
        }
    """
    # ── Fallback 1: source file missing ──────────────────────────────────────
    if not os.path.exists(clean_wav_path):
        return {
            "plate"  : plate,
            "encoded": [],
            "skipped": [],
            "failed" : [{"file": "source", "error": f"Clean WAV not found: '{clean_wav_path}'"}]
        }

    # ── Fallback 2: source file unreadable ────────────────────────────────────
    try:
        clean_signal, sr = sf.read(clean_wav_path, dtype="float32")
    except Exception as e:
        return {
            "plate"  : plate,
            "encoded": [],
            "skipped": [],
            "failed" : [{"file": "source", "error": f"Failed to read WAV: {e}"}]
        }

    # ── Fallback 3: sample rate mismatch ─────────────────────────────────────
    if sr != SAMPLE_RATE:
        return {
            "plate"  : plate,
            "encoded": [],
            "skipped": [],
            "failed" : [{"file": "source", "error": f"Sample rate {sr} Hz — expected {SAMPLE_RATE} Hz"}]
        }

    n_samples                = len(clean_signal)
    encoded, skipped, failed = [], [], []

    for profile, fn in SYNTHESISERS.items():
        for db in DB_LEVELS:
            fname = f"{plate}_{profile}_{db}db.wav"
            fpath = os.path.join(noisy_dir, fname)

            # ── Skip logic: file already exists ──────────────────────────────
            if not needs_generation(plate, profile, db, noisy_dir):
                skipped.append(fname)
                continue

            # ── Fallback 4: synthesis or write failure ────────────────────────
            try:
                noise = fn(n_samples, SAMPLE_RATE, rng)
                mixed = mix_noise(clean_signal, noise, db)
                sf.write(fpath, mixed, SAMPLE_RATE, subtype="FLOAT")
                encoded.append(fname)
            except Exception as e:
                failed.append({"file": fname, "error": str(e)})

    return {"plate": plate, "encoded": encoded, "skipped": skipped, "failed": failed}

## Step 7: Batch Runner

`generate_all()` scans `acoustic_signals/` for all `*_acoustic_id.wav` files, extracts the plate number from each filename, and calls `generate_noisy_files()` for each vehicle.

A single `rng` instance is shared across all vehicles to maintain reproducibility — the same seed always produces the same noise files regardless of how many vehicles are in the database.

If `acoustic_signals/` is empty or does not exist, a clear message is printed rather than an error.

In [ ]:
def generate_all(
    signals_dir : str,
    noisy_dir   : str
) -> dict:
    """
    Generate noisy WAV files for all vehicles found in acoustic_signals/.

    Scans for files matching *_acoustic_id.wav, extracts plate numbers,
    and processes each vehicle. Already-existing files are skipped.

    Args:
        signals_dir (str): Path to acoustic_signals/ folder (Module 2 output)
        noisy_dir   (str): Path to noisy_signals/ output folder

    Returns:
        dict: {
            "vehicles_found"  : int,
            "total_encoded"   : int,
            "total_skipped"   : int,
            "total_failed"    : int,
            "per_vehicle"     : list of per-vehicle result dicts
        }
    """
    # ── Fallback: source folder missing or empty ──────────────────────────────
    if not os.path.exists(signals_dir):
        print(f"Source folder not found: '{signals_dir}'")
        print("Run Module 2 first to generate clean WAV files.")
        return {"vehicles_found": 0, "total_encoded": 0,
                "total_skipped": 0, "total_failed": 0, "per_vehicle": []}

    clean_wavs = sorted(glob.glob(os.path.join(signals_dir, "*_acoustic_id.wav")))

    if not clean_wavs:
        print(f"No *_acoustic_id.wav files found in '{signals_dir}'")
        print("Run Module 2 first to generate clean WAV files.")
        return {"vehicles_found": 0, "total_encoded": 0,
                "total_skipped": 0, "total_failed": 0, "per_vehicle": []}

    # Single shared RNG — fixed seed ensures reproducibility across all vehicles
    rng = np.random.default_rng(SEED)

    per_vehicle                              = []
    total_encoded, total_skipped, total_failed = 0, 0, 0

    for wav_path in clean_wavs:
        # Extract plate from filename: MH12AB1234_acoustic_id.wav -> MH12AB1234
        basename = os.path.basename(wav_path)
        plate    = basename.replace("_acoustic_id.wav", "")

        result   = generate_noisy_files(plate, wav_path, noisy_dir, rng)
        per_vehicle.append(result)
        total_encoded += len(result["encoded"])
        total_skipped += len(result["skipped"])
        total_failed  += len(result["failed"])

    return {
        "vehicles_found" : len(clean_wavs),
        "total_encoded"  : total_encoded,
        "total_skipped"  : total_skipped,
        "total_failed"   : total_failed,
        "per_vehicle"    : per_vehicle
    }

## Step 8: Run

Processes all vehicles found in `acoustic_signals/`. Safe to re-run — existing files are skipped.

In [ ]:
results = generate_all(signals_dir, noisy_dir)

print(f"Vehicles found     : {results['vehicles_found']}")
print(f"Files encoded      : {results['total_encoded']}")
print(f"Files skipped      : {results['total_skipped']}")
print(f"Failures           : {results['total_failed']}")

if results["vehicles_found"] > 0:
    print()
    print(f"Per-vehicle summary:")
    hdr = f"  {'Plate':<14}  {'Encoded':>8}  {'Skipped':>8}  {'Failed':>8}"
    print(hdr)
    print(f"  {'-'*14}  {'-'*8}  {'-'*8}  {'-'*8}")
    for v in results["per_vehicle"]:
        print(f"  {v['plate']:<14}  {len(v['encoded']):>8}  {len(v['skipped']):>8}  {len(v['failed']):>8}")

if results["total_failed"] > 0:
    print()
    print("Failed files:")
    for v in results["per_vehicle"]:
        for f in v["failed"]:
            print(f"  [{v['plate']}] {f['file']} — {f['error']}")

## Step 9: Verification Utility

`verify_noisy_dir()` checks completeness and readability of all generated files. Call manually before running Module 3 to confirm the test dataset is intact.

Reports per vehicle: how many of the expected 24 files exist, which are missing, and whether every existing file loads without error.

In [ ]:
def verify_noisy_dir(signals_dir: str, noisy_dir: str) -> None:
    """
    Verify completeness and readability of all noisy WAV files.
    Call manually before running Module 3.

    Args:
        signals_dir (str): Path to acoustic_signals/ — used to find plate list
        noisy_dir   (str): Path to noisy_signals/ — files to verify
    """
    clean_wavs = sorted(glob.glob(os.path.join(signals_dir, "*_acoustic_id.wav")))
    if not clean_wavs:
        print("No clean WAV files found — run Module 2 first.")
        return

    expected_per_vehicle = len(SYNTHESISERS) * len(DB_LEVELS)
    total_expected       = len(clean_wavs) * expected_per_vehicle
    total_present        = 0
    total_missing        = 0
    total_unreadable     = 0

    print(f"Verifying {len(clean_wavs)} vehicles × {expected_per_vehicle} files = {total_expected} expected")
    print()

    for wav_path in clean_wavs:
        plate   = os.path.basename(wav_path).replace("_acoustic_id.wav", "")
        missing = []
        unreadable = []

        for profile in SYNTHESISERS:
            for db in DB_LEVELS:
                fname = f"{plate}_{profile}_{db}db.wav"
                fpath = os.path.join(noisy_dir, fname)
                if not os.path.exists(fpath):
                    missing.append(fname)
                    continue
                try:
                    w, sr = sf.read(fpath, dtype="float32")
                    assert sr == SAMPLE_RATE
                    assert w.dtype == np.float32
                    assert not np.any(np.isnan(w))
                    assert not np.any(np.isinf(w))
                    total_present += 1
                except Exception as e:
                    unreadable.append(f"{fname}: {e}")

        status = "OK" if not missing and not unreadable else "INCOMPLETE"
        print(f"  {plate:<14} {expected_per_vehicle - len(missing):>3}/{expected_per_vehicle} files  [{status}]")
        if missing:
            print(f"    Missing ({len(missing)}): {missing[:4]}{'...' if len(missing) > 4 else ''}")
        if unreadable:
            for u in unreadable:
                print(f"    Unreadable: {u}")
        total_missing    += len(missing)
        total_unreadable += len(unreadable)

    print()
    print(f"Total present    : {total_present} / {total_expected}")
    print(f"Total missing    : {total_missing}")
    print(f"Total unreadable : {total_unreadable}")
    print()
    if total_missing == 0 and total_unreadable == 0:
        print("All files verified. noisy_signals/ is ready for Module 3.")
    else:
        print("Incomplete dataset. Re-run Step 8 to generate missing files.")